In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split

titanic = sns.load_dataset('titanic')

feature_names = ['pclass', 'female', 'age', 'fare']
titanic['female'] = titanic['sex'].map({'male': 0, 'female': 1})
titanic.dropna(subset=feature_names, inplace=True)  #891 para 714

X = titanic[feature_names].to_numpy()
y = titanic['survived'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    random_state=123)

In [5]:
print('Tamanho de X_train: ', X_train.shape)
print('Tamanho de X_test: ', X_test.shape)
print('Tamanho de y_train: ', y_train.shape)
print('Tamanho de y_test: ', y_test.shape)

Tamanho de X_train:  (535, 4)
Tamanho de X_test:  (179, 4)
Tamanho de y_train:  (535,)
Tamanho de y_test:  (179,)


In [6]:
class ClassBin(nn.Module):
    # Construtor
    def __init__(self):
        super(ClassBin, self).__init__()
        self.linear1 = nn.Linear(4, 4)    # primeira hidden layer
        self.dropout1 = nn.Dropout(0.2)   # dropout layer
        self.linear2 = nn.Linear(4, 4)    # segunda hidden layer
        self.dropout2 = nn.Dropout(0.2)   # dropout layer
        self.linear3 = nn.Linear(4, 1)    # terceira hidden layer
        self.dropout3 = nn.Dropout(0.2)   # dropout layer
        self.sigmoid = nn.Sigmoid()

    # Propagação (Feed Forward)
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout1(x)
        x = F.relu(self.linear2(x))
        x = self.dropout2(x)
        x = F.relu(self.linear3(x))
        x = self.dropout3(x)
        x = self.sigmoid(x)
        return x

model = ClassBin()
print(model)

ClassBin(
  (linear1): Linear(in_features=4, out_features=4, bias=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (linear2): Linear(in_features=4, out_features=4, bias=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (linear3): Linear(in_features=4, out_features=1, bias=True)
  (dropout3): Dropout(p=0.2, inplace=False)
  (sigmoid): Sigmoid()
)


In [7]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 32  # X_train 535 / 32 = 16.71 (então são 17 batches de 32)
learning_rate = 0.1

# Instânciar o Otimizador Adam
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [8]:
from torch.utils.data import TensorDataset, DataLoader

# Converter X e y para torch.Tensor
X_train = torch.Tensor(X_train)
y_train = torch.Tensor(y_train)
X_test = torch.Tensor(X_test)
y_test = torch.Tensor(y_test)

# Um Dataset de Tensores - Array [X, y]
train = TensorDataset(X_train, y_train)
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)

test = TensorDataset(X_test, y_test)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=True)

In [9]:
for t in range(epochs):
    model.train() # Colocar o modelo em modo de treinamento (calcula os gradientes)

    # Batch Size
    for data in train_loader:
        # dar nome aos bois
        X = data[0]
        y = data[1]

        # Propagação (Feed Forward)
        y_pred = model(X)

        # Calcular erro usando a função-custo
        # y precisa virar um Tensor com tamanho (batch_size, 1)
        loss = loss_fn(y_pred, y.unsqueeze_(1))

        # Zera os gradientes antes da Retro-propagação (Backpropagation)
        model.zero_grad()

        # Retro-propagação (Backpropagation)
        loss.backward()

        # Atualização dos parâmetros
        optimizer.step()

    # Fim da Época
    print(f"""Época {t + 1},
          Custo Treino: {round(loss.item(), 3)}""")

Época 1,
          Custo Treino: 0.693
Época 2,
          Custo Treino: 0.693
Época 3,
          Custo Treino: 0.693
Época 4,
          Custo Treino: 0.693
Época 5,
          Custo Treino: 0.693
Época 6,
          Custo Treino: 0.693
Época 7,
          Custo Treino: 0.693
Época 8,
          Custo Treino: 0.693
Época 9,
          Custo Treino: 0.693
Época 10,
          Custo Treino: 0.693
Época 11,
          Custo Treino: 0.693
Época 12,
          Custo Treino: 0.693
Época 13,
          Custo Treino: 0.693
Época 14,
          Custo Treino: 0.693
Época 15,
          Custo Treino: 0.693
Época 16,
          Custo Treino: 0.693
Época 17,
          Custo Treino: 0.693
Época 18,
          Custo Treino: 0.693
Época 19,
          Custo Treino: 0.693
Época 20,
          Custo Treino: 0.693
Época 21,
          Custo Treino: 0.693
Época 22,
          Custo Treino: 0.693
Época 23,
          Custo Treino: 0.693
Época 24,
          Custo Treino: 0.693
Época 25,
          Custo Treino: 0.693
Época 26,

### Acurácia do Modelo

Usar o comando `model.eval()`

Para a métrica acurácia, retorna um score de acurácia `float` entre $0$ e $1$
    
> Obs: Regressão Logística acurácias: 0.69 Treino e 0.7 Teste

> Obs: *Support Vector Machines* acurácias: 0.79 Treino e 0.75 Teste

> Obs: Árvores de Decisão acurácias: 0.79 Treino e 0.79 Teste

> Obs: Florestas Aleatórias acurácias: 0.84 Treino e 0.82 Teste

In [10]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.590654194355011

 ---------------------------

Acurácia de Teste: 0.6033519506454468


# Exercicios

**1) Com base no material apresentado no notebook, o que é uma função de
ativação, como a ReLU? Por que normalmente usamos funções de ativação entre as camadas?**

**R**

Uma função de ativação é uma expressão matemática aplicada ao resultado de cada neurônio em uma rede neural. Ela atua sobre a soma ponderada das entradas, definindo o valor final produzido pelo neurônio e decidindo se ele será “ativado” para transmitir informações à próxima camada. Um exemplo é a ReLU (Rectified Linear Unit), que devolve o próprio valor de entrada quando ele é positivo e, caso contrário, retorna zero (f(x) = max(0, x)).

O principal motivo para utilizar funções de ativação — especialmente as não lineares, como a ReLU — é inserir não linearidade no modelo. Sem esse recurso, mesmo redes neurais muito profundas se comportariam como um modelo de regressão linear, já que a combinação de várias transformações lineares continua sendo linear. A presença de não linearidade permite que a rede aprenda e represente relações muito mais complexas entre entradas e saídas, algo fundamental em aplicações reais, como reconhecimento de imagens e processamento de linguagem natural.


**2. Explique o que cada uma das seguintes linhas de código faz e por que ela é necessária:**


*   model.train()
*   optimizer.step()

**Além disso, explique:
Qual é a diferença fundamental entre os modos model.train() e model.eval()?**

**R**

*   model.train(): Essa instrução coloca o modelo em modo de treinamento. Isso é essencial porque algumas camadas, como Dropout e BatchNorm, mudam seu comportamento dependendo do modo. Durante o treinamento, o Dropout desliga aleatoriamente certos neurônios para reduzir o overfitting, enquanto o BatchNorm ajusta os dados com base nas estatísticas do lote atual. Por isso, é importante ativar esse modo antes de iniciar o treinamento, garantindo que essas camadas funcionem corretamente.

*   optimizer.step(): Essa função é responsável por atualizar os parâmetros do modelo (pesos e vieses) a partir dos gradientes calculados na etapa de retropropagação (loss.backward()). O método de otimização utilizado define como essa atualização acontece, levando em conta a taxa de aprendizado e outros hiperparâmetros, ajustando os parâmetros na direção que reduz a função de perda.

**Diferença entre model.train() e model.eval()**

model.train(): Coloca o modelo em modo de treinamento. Nesse estado, camadas como Dropout e BatchNorm operam de forma apropriada para o aprendizado — aplicando desligamento aleatório de neurônios e atualizando estatísticas de normalização por lote. Além disso, o PyTorch mantém o rastreamento de gradientes, necessário para realizar a retropropagação.

model.eval(): Define o modelo em modo de avaliação. Nesse caso, o Dropout é desativado e o BatchNorm passa a usar estatísticas já aprendidas durante o treinamento, em vez de calcular novas. Também não há necessidade de calcular gradientes, o que reduz o consumo de memória e melhora a eficiência durante a inferência.

**3. Modifique a classe ClassBin para que a rede tenha a seguinte arquitetura:**
*   Camada de entrada: mantém as 4 features de entrada
*   Primeira camada oculta: nn.Linear com 4 neurônios de entrada e 16
neurônios de saída, seguida por ativação ReLU
*   Segunda camada oculta: nn.Linear com 16 neurônios de entrada e 8
neurônios de saída, seguida por ativação ReLU
*   Camada de saída: nn.Linear com 8 neurônios de entrada e 1 neurônio de
saída, seguida por ativação Sigmoid
*   Remova todas as camadas de Dropout da nova arquitetura

Depois:
*   Treine o novo modelo com os mesmos hiperparâmetros do modelo original
*   Compare a acurácia final de treino e teste com a do modelo original


In [11]:
class ClassBin2(nn.Module):
    def __init__(self):
        super(ClassBin, self).__init__()

        self.linear1 = nn.Linear(4, 16) # primeira hidden layer  com 4 neurônios de entrada e 16 neurônios de saída
        self.linear2 = nn.Linear(16, 8) # segunda hidden layer # segunda hidden layer
        self.linear3 = nn.Linear(8, 1)  # terceira hidden layer om 8 neurônios de entrada e 1 neurônio de saída
        self.sigmoid = nn.Sigmoid()
        self.ReLu = nn.ReLU()

    # Propagação (Feed Forward)
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = F.relu(self.linear3(x))
        x = self.sigmoid(x)
        return x

model = ClassBin()
print(model)

ClassBin(
  (linear1): Linear(in_features=4, out_features=4, bias=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (linear2): Linear(in_features=4, out_features=4, bias=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (linear3): Linear(in_features=4, out_features=1, bias=True)
  (dropout3): Dropout(p=0.2, inplace=False)
  (sigmoid): Sigmoid()
)


In [12]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.672897219657898

 ---------------------------

Acurácia de Teste: 0.6424580812454224


**Modelo Original**

Acurácia de Treino: 0.590654194355011

Acurácia de Teste: 0.6033519506454468

**Modelo 2.0 - EX3:**

Acurácia de Treino: 0.672897219657898

Acurácia de Teste: 0.6424580812454224

**4. Usando o modelo original do notebook:**
*   Substitua o otimizador Adam por SGD
*   Treine o modelo com o novo otimizador
*   Observe o comportamento do custo (loss) durante o treinamento
*   Compare a acurácia final com a obtida anteriormente

**Responda:
O SGD com essa taxa de aprendizado pareceu uma boa escolha?**

In [13]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 32  # X_train 535 / 32 = 16.71 (então são 17 batches de 32)
learning_rate = 0.1

# Instânciar o Otimizador Adam
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [16]:
for t in range(epochs):
    model.train() # Colocar o modelo em modo de treinamento (calcula os gradientes)

    # Batch Size
    for data in train_loader:
        # dar nome aos bois
        X = data[0]
        y = data[1]

        # Propagação (Feed Forward)
        y_pred = model(X)

        # Calcular erro usando a função-custo
        # y precisa virar um Tensor com tamanho (batch_size, 1)
        loss = loss_fn(y_pred, y.unsqueeze_(1))

        # Zera os gradientes antes da Retro-propagação (Backpropagation)
        model.zero_grad()

        # Retro-propagação (Backpropagation)
        loss.backward()

        # Atualização dos parâmetros
        optimizer.step()

    # Fim da Época
    print(f"""Época {t + 1},
          Custo Treino: {round(loss.item(), 3)}""")

Época 1,
          Custo Treino: 0.693
Época 2,
          Custo Treino: 0.693
Época 3,
          Custo Treino: 0.693
Época 4,
          Custo Treino: 0.693
Época 5,
          Custo Treino: 0.693
Época 6,
          Custo Treino: 0.693
Época 7,
          Custo Treino: 0.693
Época 8,
          Custo Treino: 0.693
Época 9,
          Custo Treino: 0.693
Época 10,
          Custo Treino: 0.693
Época 11,
          Custo Treino: 0.693
Época 12,
          Custo Treino: 0.693
Época 13,
          Custo Treino: 0.693
Época 14,
          Custo Treino: 0.693
Época 15,
          Custo Treino: 0.693
Época 16,
          Custo Treino: 0.693
Época 17,
          Custo Treino: 0.693
Época 18,
          Custo Treino: 0.693
Época 19,
          Custo Treino: 0.693
Época 20,
          Custo Treino: 0.693
Época 21,
          Custo Treino: 0.693
Época 22,
          Custo Treino: 0.693
Época 23,
          Custo Treino: 0.693
Época 24,
          Custo Treino: 0.693
Época 25,
          Custo Treino: 0.693
Época 26,

In [17]:
model.eval()

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.590654194355011

 ---------------------------

Acurácia de Teste: 0.6033519506454468


**Resposta 4:**

Adam:

*   Acurácia de Treino: 0.590654194355011
*   Acurácia de Teste: 0.6033519506454468

SGD:

*   Acurácia de Treino: 0.590654194355011

*   Acurácia de Teste: 0.6033519506454468

Com o uso do otimizador SGD, observou-se que o custo de treinamento apresentou variações nas primeiras épocas, porém rapidamente passou a se estabilizar, indicando pouca evolução no processo de aprendizado após certo ponto.

Além disso, a acurácia de treino e de teste permaneceu praticamente inalterada em relação ao modelo treinado com Adam, sugerindo que o modelo não conseguiu melhorar seu desempenho com o novo otimizador.

Isso indica que, para essa configuração de taxa de aprendizado, o SGD não foi uma escolha eficaz, pois não proporcionou uma redução consistente do erro nem ganhos em acurácia.

**5. Usando o modelo original e o otimizador Adam:**
*   No DataLoader, mude o batch_size de 32 para um valor muito maior, como
512
*   Treine o modelo e observe a acurácia
*   Depois, mude o batch_size para um valor bem pequeno, como 4, e treine
novamente

**Responda:
Como a mudança no batch_size afetou:**
*   a estabilidade do custo (loss) a cada época
*   a acurácia final do modelo

**Batch_size 512**

In [23]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 512
learning_rate = 0.1

# reinicia o modelo do zero
model = ClassBin()

# recria os loaders com o novo batch_size
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=True)

# Adam
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [24]:
for t in range(epochs):
    model.train()

    for data in train_loader:
        X = data[0]
        y = data[1]

        y_pred = model(X)
        loss = loss_fn(y_pred, y.unsqueeze(1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Época {t + 1}, Custo Treino: {loss.item():.6f}")

Época 1, Custo Treino: 0.693147
Época 2, Custo Treino: 0.693147
Época 3, Custo Treino: 0.693147
Época 4, Custo Treino: 0.693147
Época 5, Custo Treino: 0.693147
Época 6, Custo Treino: 0.693147
Época 7, Custo Treino: 0.693147
Época 8, Custo Treino: 0.693147
Época 9, Custo Treino: 0.693147
Época 10, Custo Treino: 0.693147
Época 11, Custo Treino: 0.693147
Época 12, Custo Treino: 0.693147
Época 13, Custo Treino: 0.693147
Época 14, Custo Treino: 0.693147
Época 15, Custo Treino: 0.693147
Época 16, Custo Treino: 0.693147
Época 17, Custo Treino: 0.693147
Época 18, Custo Treino: 0.693147
Época 19, Custo Treino: 0.693147
Época 20, Custo Treino: 0.693147
Época 21, Custo Treino: 0.693147
Época 22, Custo Treino: 0.693147
Época 23, Custo Treino: 0.693147
Época 24, Custo Treino: 0.693147
Época 25, Custo Treino: 0.693147
Época 26, Custo Treino: 0.693147
Época 27, Custo Treino: 0.693147
Época 28, Custo Treino: 0.693147
Época 29, Custo Treino: 0.693147
Época 30, Custo Treino: 0.693147
Época 31, Custo Tre

In [25]:
model.eval()

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.590654194355011
Acurácia de Teste: 0.6033519506454468


**Batch_size 4**

In [26]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 4 # X_train 535 / 32 = 16.71 (então são 17 batches de 32)
learning_rate = 0.1

# Instânciar o Otimizador Adam
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [27]:
for t in range(epochs):
    model.train()

    for data in train_loader:
        X = data[0]
        y = data[1]

        y_pred = model(X)
        loss = loss_fn(y_pred, y.unsqueeze(1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Época {t + 1}, Custo Treino: {loss.item():.6f}")

Época 1, Custo Treino: 0.693147
Época 2, Custo Treino: 0.693147
Época 3, Custo Treino: 0.693147
Época 4, Custo Treino: 0.693147
Época 5, Custo Treino: 0.693147
Época 6, Custo Treino: 0.693147
Época 7, Custo Treino: 0.693147
Época 8, Custo Treino: 0.693147
Época 9, Custo Treino: 0.693147
Época 10, Custo Treino: 0.693147
Época 11, Custo Treino: 0.693147
Época 12, Custo Treino: 0.693147
Época 13, Custo Treino: 0.693147
Época 14, Custo Treino: 0.693147
Época 15, Custo Treino: 0.693147
Época 16, Custo Treino: 0.693147
Época 17, Custo Treino: 0.693147
Época 18, Custo Treino: 0.693147
Época 19, Custo Treino: 0.693147
Época 20, Custo Treino: 0.693147
Época 21, Custo Treino: 0.693147
Época 22, Custo Treino: 0.693147
Época 23, Custo Treino: 0.693147
Época 24, Custo Treino: 0.693147
Época 25, Custo Treino: 0.693147
Época 26, Custo Treino: 0.693147
Época 27, Custo Treino: 0.693147
Época 28, Custo Treino: 0.693147
Época 29, Custo Treino: 0.693147
Época 30, Custo Treino: 0.693147
Época 31, Custo Tre

In [28]:
model.eval()

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x: 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.590654194355011
Acurácia de Teste: 0.6033519506454468


**R**

No experimento realizado, a alteração do batch_size para um valor muito maior (512) e depois para um valor muito menor (4) não causou mudanças significativas no comportamento do custo ao longo das épocas. O loss permaneceu estável e a acurácia final também se manteve praticamente igual nos dois casos.

Assim, para este modelo e para esta base de dados, a variação do batch_size não teve influência relevante no desempenho final.